#### Aqbil Gradiansyah
#### 10522138
#### Linear Regression 
#### Dataset The Soko CTC 2024-2026
## Import Library yang akan digunakan

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import export_text
import warnings
import joblib
warnings.filterwarnings('ignore')

## Import Dataset dan Gabungkan karena dataset makanan & minuman terpisah

In [2]:
df_makanan = pd.read_excel('Dataset/Daily Sales Menu Makanan.xlsx', skiprows=13)
df_minuman = pd.read_excel('Dataset/Daily Sales Menu Minuman.xlsx', skiprows=13)

df = pd.concat([df_makanan, df_minuman], ignore_index=True)
print(df.shape)

(19542, 14)


## Seleksi Atribut yang akan digunakan untuk training

In [3]:
# Pilih hanya kolom numerik yang relevan
kolom_numerik = ['Qty', 'Subtotal', 'Service Charge', 'Tax Total', 'VAT Total', 'Total']

# Pastikan semua kolom itu ada di df
kolom_ada = [col for col in kolom_numerik if col in df.columns]

# Hitung korelasi
corr_result = df[kolom_ada].corr()
print(corr_result['Qty'])

Qty               1.000000
Subtotal          0.875142
Service Charge    0.865820
Tax Total         0.865820
VAT Total              NaN
Total             0.874650
Name: Qty, dtype: float64


In [4]:
df = df[['Sales Date', 'Menu Name', 'Type', 'Qty']]

df['Menu Name'] = df['Menu Name'].str.strip()
df['Sales Date'] = pd.to_datetime(df['Sales Date'])
df['Qty'] = pd.to_numeric(df['Qty'], errors='coerce')
df.info

<bound method DataFrame.info of       Sales Date               Menu Name       Type  Qty
0     2024-01-01              Choco lava  Ala Carte    1
1     2024-01-01  Chocolatte banana roll  Ala Carte    1
2     2024-01-01            Cireng Bumbu  Ala Carte    4
3     2024-01-01            French Fries  Ala Carte    2
4     2024-01-01  FF & Sausage Bratwurst  Ala Carte    3
...          ...                     ...        ...  ...
19537 2026-01-01           Lemon Tea Ice  Ala Carte    3
19538 2026-01-01          Lychee Ice Tea  Ala Carte    4
19539 2026-01-01      Naga Black Tea Hot  Ala Carte    3
19540 2026-01-01      Naga Black Tea Ice  Ala Carte    5
19541 2026-01-01            Thai Tea Ice  Ala Carte    3

[19542 rows x 4 columns]>

## Cek Missing value dan Duplicate

In [5]:
print("Missing values:\n", df.isnull().sum())
print("\nDuplicates (raw):", df.duplicated().sum())
print("\nNilai unik kolom Type:\n", df['Type'].value_counts())

# Cek duplikat detail
duplicates_all = df[df.duplicated(keep=False)]
duplicates_all.sort_values(by=list(df.columns))

# Deteksi outlier pada kolom Qty menggunakan metode IQR
Q1 = df['Qty'].quantile(0.25)
Q3 = df['Qty'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Qty'] < lower_bound) | (df['Qty'] > upper_bound)]

print(f"\nDeteksi Outlier pada kolom Qty (metode IQR):")
print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Batas bawah: {lower_bound}, Batas atas: {upper_bound}")
print(f"Jumlah outlier terdeteksi: {len(outliers)}")
print(f"Persentase outlier: {len(outliers)/len(df)*100:.2f}%")
print("\nSampel outlier:")
print(outliers[['Sales Date', 'Menu Name', 'Qty', 'Type']].head(10))

Missing values:
 Sales Date    0
Menu Name     0
Type          0
Qty           0
dtype: int64

Duplicates (raw): 10

Nilai unik kolom Type:
 Type
Ala Carte    19516
Free Item       26
Name: count, dtype: int64

Deteksi Outlier pada kolom Qty (metode IQR):
Q1: 1.0, Q3: 2.0, IQR: 1.0
Batas bawah: -0.5, Batas atas: 3.5
Jumlah outlier terdeteksi: 2478
Persentase outlier: 12.68%

Sampel outlier:
   Sales Date                  Menu Name  Qty       Type
2  2024-01-01               Cireng Bumbu    4  Ala Carte
13 2024-01-02               Cireng Bumbu    4  Ala Carte
48 2024-01-06            Tahu Lada Garam    4  Ala Carte
67 2024-01-08    Chicken Zurich Mushroom    6  Ala Carte
79 2024-01-13               Cireng Bumbu    6  Ala Carte
80 2024-01-13               French Fries    4  Ala Carte
82 2024-01-13          Soko Mix Platters    4  Ala Carte
83 2024-01-13            Tahu Lada Garam    4  Ala Carte
85 2024-01-13  Carbonara pasta spaghetti    4  Ala Carte
90 2024-01-13         Chicken Lemong

## Agregasi Harian

In [6]:
df = df.drop(columns=['Type'])

df_agg = df.groupby(['Sales Date', 'Menu Name'])['Qty'].sum().reset_index()
df_agg.columns = ['date', 'menu', 'qty']
print(df_agg.shape)
df_agg.head(10)

(19500, 3)


,date,menu,qty
0,2024-01-01,Aglio olio pasta spaghetti,1
1,2024-01-01,Bolognais Pasta Spaghetti,1
2,2024-01-01,Carbonara pasta spaghetti,1
3,2024-01-01,Charcoal Latte Ice,2
4,2024-01-01,Choco Java Hot,3
5,2024-01-01,Choco Java Ice,3
6,2024-01-01,Choco lava,1
7,2024-01-01,Chocolatte banana roll,1
8,2024-01-01,Cireng Bumbu,4
9,2024-01-01,Es Kopi Soko,9


## Date gap filling

In [7]:
all_dates = pd.date_range(df_agg['date'].min(), df_agg['date'].max())
all_menus = df_agg['menu'].unique()

full_index = pd.MultiIndex.from_product([all_dates, all_menus], names=['date', 'menu'])
df_full = df_agg.set_index(['date', 'menu']).reindex(full_index, fill_value=0).reset_index()

print(df_full.shape)
df_full.tail(10)

(132492, 3)


,date,menu,qty
132482,2026-01-01,Tiramisu Latte Hot,0
132483,2026-01-01,Nasi Gepuk,0
132484,2026-01-01,Kopi Jahe Panas,0
132485,2026-01-01,Req american breakfast + mineral,0
132486,2026-01-01,Req chicken blackpepper+Teh,0
132487,2026-01-01,Black Lemon,3
132488,2026-01-01,Black Raspberry,1
132489,2026-01-01,Black apple,0
132490,2026-01-01,Bundling Yoga,0
132491,2026-01-01,Popcorn,4


## Feature Engineering

In [8]:
df_full = df_full.sort_values(['menu', 'date']).reset_index(drop=True)

df_full['day_of_week'] = df_full['date'].dt.dayofweek
df_full['month'] = df_full['date'].dt.month
df_full['lag_1'] = df_full.groupby('menu')['qty'].shift(1)
df_full['lag_7'] = df_full.groupby('menu')['qty'].shift(7)
df_full['is_weekend'] = df_full['day_of_week'].isin([5, 6]).astype(int)
df_full['lag_3'] = df_full.groupby('menu')['qty'].shift(3)
df_full['lag_14'] = df_full.groupby('menu')['qty'].shift(14)
df_full['week_of_month'] = df_full['date'].dt.day // 7 + 1
df_full['rolling_14'] = df_full.groupby('menu')['qty'].transform(
    lambda x: x.shift(1).rolling(14).mean()
)
# Rolling std (volatilitas permintaan)
df_full['rolling_std_7'] = df_full.groupby('menu')['qty'].transform(
    lambda x: x.shift(1).rolling(7).std()
)
df_full['rolling_7'] = df_full.groupby('menu')['qty'].transform(
    lambda x: x.shift(1).rolling(7).mean()
)

df_full = df_full.dropna()
print(df_full.shape)
df_full.head()

(129958, 14)


,date,menu,qty,day_of_week,month,lag_1,lag_7,is_weekend,lag_3,lag_14,week_of_month,rolling_14,rolling_std_7,rolling_7
14,2024-01-15,Aeropress Import,0,0,1,0.0,0.0,0,0.0,0.0,3,0.0,0.0,0.0
15,2024-01-16,Aeropress Import,0,1,1,0.0,0.0,0,0.0,0.0,3,0.0,0.0,0.0
16,2024-01-17,Aeropress Import,0,2,1,0.0,0.0,0,0.0,0.0,3,0.0,0.0,0.0
17,2024-01-18,Aeropress Import,0,3,1,0.0,0.0,0,0.0,0.0,3,0.0,0.0,0.0
18,2024-01-19,Aeropress Import,0,4,1,0.0,0.0,0,0.0,0.0,3,0.0,0.0,0.0


## Split train/test 80:20

In [9]:
df_full = df_full.sort_values('date').reset_index(drop=True)

split_idx = int(len(df_full) * 0.8)
split_date = df_full.iloc[split_idx]['date']
print("Split date:", split_date)

train = df_full[df_full['date'] < split_date]
test  = df_full[df_full['date'] >= split_date]

print("Train:", train.shape, "| Test:", test.shape)

Split date: 2025-08-11 00:00:00
Train: (103894, 14) | Test: (26064, 14)


## Training

In [10]:
# LR Baseline (5 fitur)
features_v1 = ['day_of_week', 'month', 'lag_1', 'lag_7', 'rolling_7']

X_train_v1 = train[features_v1]
X_test_v1 = test[features_v1]

# Mendefinisikan target prediksi
y_train = train['qty']
y_test = test['qty']

from sklearn.linear_model import LinearRegression
model_lr_v1 = LinearRegression()
model_lr_v1.fit(X_train_v1, y_train)
y_pred_lr_v1 = model_lr_v1.predict(X_test_v1)

print("LR v1 - Baseline (5 fitur)")
print(f"MAE  : {mean_absolute_error(y_test, y_pred_lr_v1):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, y_pred_lr_v1)):.4f}")
print(f"R²   : {r2_score(y_test, y_pred_lr_v1):.4f}")

LR v1 - Baseline (5 fitur)
MAE  : 0.4098
RMSE : 0.9376
R²   : 0.4139


In [11]:
# LR v2 (11 fitur)
features = ['day_of_week', 'month', 'lag_1', 'lag_3', 'lag_7',
            'lag_14', 'rolling_7', 'rolling_14', 'rolling_std_7',
            'is_weekend', 'week_of_month']

X_train = train[features]
y_train = train['qty']
X_test  = test[features]
y_test  = test['qty']

model_lr = LinearRegression()
model_lr.fit(X_train, y_train)
print("Training selesai.")

Training selesai.


In [12]:
from sklearn.tree import export_text
import pandas as pd

# Define feature_names
feature_names = X_train.columns.tolist()

# ===== CARI SAMPEL LR YANG AKTUALNYA > 0 =====
# Cari index di test set yang aktualnya ada penjualan
idx_bagus = y_test[y_test > 0].index[5]  # ambil sampel ke-5 yang aktual > 0
sampel = X_test.loc[idx_bagus]
aktual = y_test.loc[idx_bagus]

# Hitung manual LR
kontribusi = []
for fitur, koef in zip(feature_names, model_lr.coef_):
    kontribusi.append({
        'Fitur': fitur,
        'Nilai Input': round(sampel[fitur], 4),
        'Koefisien': round(koef, 6),
        'Kontribusi': round(sampel[fitur] * koef, 6)
    })

tabel_lr = pd.DataFrame(kontribusi)
intersep_val = round(model_lr.intercept_, 6)
prediksi_manual = intersep_val + tabel_lr['Kontribusi'].sum()
prediksi_model = model_lr.predict([sampel])[0]

print("TABEL PERHITUNGAN MANUAL LINEAR REGRESSION")
print(f"Intersep: {intersep_val}")
print(tabel_lr.to_string(index=False))
print(f"\nTotal Kontribusi Fitur : {tabel_lr['Kontribusi'].sum():.6f}")
print(f"Prediksi Manual (Ŷ)   : {prediksi_manual:.4f}")
print(f"Prediksi Model        : {prediksi_model:.4f}")
print(f"Aktual                : {aktual}")
print(f"Error (|Aktual - Ŷ|)  : {abs(aktual - prediksi_manual):.4f}")

TABEL PERHITUNGAN MANUAL LINEAR REGRESSION
Intersep: -0.078195
        Fitur  Nilai Input  Koefisien  Kontribusi
  day_of_week       0.0000   0.004986    0.000000
        month       8.0000   0.003966    0.031729
        lag_1       1.0000   0.122282    0.122282
        lag_3       2.0000  -0.052131   -0.104261
        lag_7       0.0000   0.147720    0.000000
       lag_14       1.0000   0.137224    0.137224
    rolling_7       1.0000   0.172318    0.172318
   rolling_14       1.0000   0.471512    0.471512
rolling_std_7       1.5275  -0.098117   -0.149877
   is_weekend       0.0000   0.175567    0.000000
week_of_month       2.0000   0.010147    0.020294

Total Kontribusi Fitur : 0.701221
Prediksi Manual (Ŷ)   : 0.6230
Prediksi Model        : 0.6230
Aktual                : 1
Error (|Aktual - Ŷ|)  : 0.3770


In [13]:
print(f"Index sampel  : {idx_bagus}")
print(f"Tanggal       : {df_full.loc[idx_bagus, 'date']}")
print(f"Menu          : {df_full.loc[idx_bagus, 'menu']}")
print(f"Aktual Qty    : {df_full.loc[idx_bagus, 'qty']}")
print(f"\nNilai fitur:")
print(X_test.loc[idx_bagus])

Index sampel  : 103969
Tanggal       : 2025-08-11 00:00:00
Menu          : Cireng Bumbu
Aktual Qty    : 1

Nilai fitur:
day_of_week      0.000000
month            8.000000
lag_1            1.000000
lag_3            2.000000
lag_7            0.000000
lag_14           1.000000
rolling_7        1.000000
rolling_14       1.000000
rolling_std_7    1.527525
is_weekend       0.000000
week_of_month    2.000000
Name: 103969, dtype: float64


## Evaluasi

In [14]:
y_pred = model_lr.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("LR v2 (11 fitur)")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

LR v2 (11 fitur)
MAE  : 0.3879
RMSE : 0.9232
R²   : 0.4317


## Save Hasil Peramalan

In [15]:
results = test[['date', 'menu']].copy()
results['y_actual'] = y_test.values
results['y_pred_lr'] = y_pred

#results.to_csv('hasil_lr.csv', index=False)
#print("Hasil disimpan ke hasil_lr.csv")
results.head(10)

,date,menu,y_actual,y_pred_lr
103894,2025-08-11,Extra Espresso,0,0.041186
103895,2025-08-11,Baby Purple,0,-0.026172
103896,2025-08-11,Iced Vanilla Latte,0,-0.026172
103897,2025-08-11,Iga Bakar,0,-0.026172
103898,2025-08-11,Sate Jando,0,-0.119984
103899,2025-08-11,Buntut Bakar,0,-0.026172
103900,2025-08-11,Donat,0,-0.026172
103901,2025-08-11,Es Kopi Soko,2,4.586578
103902,2025-08-11,Brulle Creamy Macaroni,0,-0.085251
103903,2025-08-11,Crispy Chicken Dice,0,0.689439


In [16]:
import joblib

# Simpan model rf_v2 ke dalam file bernama 'model_lr.pkl'
joblib.dump(model_lr, "model_lr_v2.pkl")

print("Model Linear Regression V2 berhasil disimpan!")

Model Linear Regression V2 berhasil disimpan!


## Eksplosi BOM

In [17]:
import pandas as pd
file = 'Dataset/SOKO - Master Menu Soko.xlsx'

# 1. LOAD MENU BOM
bom_bar     = pd.read_excel(file, sheet_name='Menu Bar', header=4)
bom_kitchen = pd.read_excel(file, sheet_name='Menu Kitchen', header=4)

bom_all = pd.concat([bom_bar, bom_kitchen], ignore_index=True)
bom_all.columns = bom_all.columns.str.strip()
bom_all = bom_all.dropna(how='all')

# forward fill menu
bom_all['Menu'] = bom_all['Menu'].ffill()

# cleaning
bom_all['Menu']  = bom_all['Menu'].astype(str).str.strip().str.lower()
bom_all['Bahan'] = bom_all['Bahan'].astype(str).str.strip().str.lower()

bom_all = bom_all.dropna(subset=['Bahan'])
bom_all['Recipe QTY'] = pd.to_numeric(bom_all['Recipe QTY'], errors='coerce').fillna(0)

bom_all = bom_all.rename(columns={
    'Menu': 'menu',
    'Bahan': 'bahan',
    'Recipe QTY': 'recipe_qty',
    'Satuan': 'satuan'
})

bom_all = bom_all[['menu','bahan','recipe_qty','satuan']]

# 2. LOAD PRODUCTION BOM (KTC)
prod_bar = pd.read_excel(file, sheet_name='Production Bar', header=4)
prod_kitchen = pd.read_excel(file, sheet_name='Production Kitchen', header=4)

prod_all = pd.concat([prod_bar, prod_kitchen], ignore_index=True)
prod_all.columns = prod_all.columns.str.strip()
prod_all = prod_all.dropna(how='all')

prod_all['Menu'] = prod_all['Menu'].ffill()

prod_all['is_yield'] = prod_all['Bahan'].isna()

prod_all['Menu']  = prod_all['Menu'].astype(str).str.strip().str.lower()
prod_all['Bahan'] = prod_all['Bahan'].astype(str).str.strip().str.lower()
prod_all['Recipe QTY'] = pd.to_numeric(prod_all['Recipe QTY'], errors='coerce').fillna(0)

# 2A. YIELD KTC
yield_df = prod_all[prod_all['is_yield']].copy()
yield_df = yield_df.rename(columns={
    'Menu': 'parent',
    'Recipe QTY': 'yield_qty'
})[['parent','yield_qty']]

# 2B. DETAIL KTC
prod_detail = prod_all[~prod_all['is_yield']].copy()
prod_detail = prod_detail.rename(columns={
    'Menu': 'parent',
    'Bahan': 'child',
    'Recipe QTY': 'qty_prod',
    'Satuan': 'satuan_prod'
})

prod_detail = prod_detail[['parent','child','qty_prod','satuan_prod']]

# 2C. NORMALISASI
prod_detail = prod_detail.merge(yield_df, on='parent', how='left')
prod_detail['qty_per_unit'] = prod_detail['qty_prod'] / prod_detail['yield_qty']

# 3. PREDIKSI LR
pred = results[['date','menu','y_pred_lr']].copy()
pred['menu'] = pred['menu'].str.lower().str.strip()
pred['y_pred_lr'] = pred['y_pred_lr'].clip(lower=0).round()

# 4. LEVEL 1: MENU → KTC
menu_join = pred.merge(bom_all, on='menu', how='left')

# 5. LEVEL 2: KTC → RAW
full_bom = menu_join.merge(
    prod_detail,
    left_on='bahan',
    right_on='parent',
    how='left'
)

# 6. FINAL CALC
full_bom['final_bahan'] = full_bom['child'].fillna(full_bom['bahan'])

full_bom['final_qty'] = (
    full_bom['y_pred_lr'] *
    full_bom['recipe_qty'] *
    full_bom['qty_per_unit'].fillna(1)
)

full_bom['final_satuan'] = full_bom['satuan_prod'].fillna(full_bom['satuan'])

# 7. AGREGASI RAW
stok_harian_lr = full_bom.groupby(
    ['date','final_bahan','final_satuan']
)['final_qty'].sum().reset_index()

stok_harian_lr.columns = ['Tanggal','Bahan','Satuan','Kebutuhan']
stok_harian_lr['Kebutuhan'] = stok_harian_lr['Kebutuhan'].round(2)

# 8. KTC USAGE
ktc_usage = menu_join.groupby(
    ['date','bahan','satuan']
).apply(lambda x: (x['y_pred_lr'] * x['recipe_qty']).sum()).reset_index(name='kebutuhan_ktc')

# 9. LABELING
ktc_usage_renamed = ktc_usage.rename(columns={
    'bahan': 'Item',
    'satuan': 'Satuan',
    'kebutuhan_ktc': 'Kebutuhan'
})
ktc_usage_renamed['Tipe'] = 'KTC'

raw_usage = stok_harian_lr.rename(columns={
    'Bahan': 'Item',
    'Satuan': 'Satuan',
    'Kebutuhan': 'Kebutuhan'
})
raw_usage['Tipe'] = 'RAW'

# 10. COMBINE
combined_usage = pd.concat([ktc_usage_renamed, raw_usage], ignore_index=True)

total_usage = combined_usage.groupby(
    ['Tanggal','Item','Satuan','Tipe']
)['Kebutuhan'].sum().reset_index()

total_usage = total_usage.sort_values(
    by=['Tanggal','Kebutuhan'],
    ascending=[True, False]
)
# urutkan ulang kolomnya
total_usage = total_usage[['Tanggal','Item','Tipe','Kebutuhan','Satuan']]

# 11. OUTPUT
stok_harian_lr.to_csv('rekomendasi_stok_harian_lr.csv', index=False)

print("\nBAHAN MENTAH")
print(stok_harian_lr.head(10))

print("\nKEBUTUHAN KTC")
print(ktc_usage.head(10))

print("\nTOTAL GABUNGAN KTC + RAW")
print(total_usage.head(20))

# 12. FILTER HARI TERTENTU
tanggal_filter = '2025-08-11'

print(f"\nDETAIL {tanggal_filter}")
print(
    total_usage[total_usage['Tanggal'] == tanggal_filter]
    .sort_values(by='Kebutuhan', ascending=False)
    .head(20)
)

# 13. DEBUG MENU SPESIFIK
full_bom[
    (full_bom['date'] == '2025-08-11') & 
    # isi nama menu disini dan lowercase
    (full_bom['menu'] == 'soko fried rice') &
    (full_bom['final_qty'] > 0)
][['menu','y_pred_lr','bahan','final_bahan','final_qty','final_satuan']] \
.sort_values(by='final_qty', ascending=False)


BAHAN MENTAH
     Tanggal                     Bahan Satuan  Kebutuhan
0 2025-08-11                 air galon     ml     923.74
1 2025-08-11   air mineral water 600ml  Botol       3.00
2 2025-08-11  arghani chocolate mixing     gr      30.00
3 2025-08-11         asam jawa kemasan     gr       0.10
4 2025-08-11        ayam boneless dada     gr     181.08
5 2025-08-11                ayam sayap     gr       0.00
6 2025-08-11             baking powder     gr       0.00
7 2025-08-11             bakso mas eko    pcs       0.00
8 2025-08-11             bawang bombay     gr      36.36
9 2025-08-11              bawang merah     gr       0.04

KEBUTUHAN KTC
        date                     bahan satuan  kebutuhan_ktc
0 2025-08-11                 air galon     ml          622.0
1 2025-08-11   air mineral water 600ml  Botol            3.0
2 2025-08-11  arghani chocolate mixing     gr           30.0
3 2025-08-11        ayam boneless dada     gr          180.0
4 2025-08-11             bakso mas eko 

,menu,y_pred_lr,bahan,final_bahan,final_qty,final_satuan
959,soko fried rice,1.0,krupuk sumber sari,krupuk sumber sari,30.000000,gr
953,soko fried rice,1.0,bawang bombay,bawang bombay,16.000000,gr
954,soko fried rice,1.0,oyster sauce panda,oyster sauce panda,5.000000,gr
956,soko fried rice,1.0,merica,merica,2.000000,gr
955,soko fried rice,1.0,powder chicken,powder chicken,2.000000,gr
943,soko fried rice,1.0,ktc pro rice,beras rojo lele,1.000000,gr
952,soko fried rice,1.0,telur ayam,telur ayam,1.000000,pcs
960,soko fried rice,1.0,lunch box ukuran m,lunch box ukuran m,1.000000,pcs
945,soko fried rice,1.0,ktc pro mix vegetable,wortel grade a,0.571429,gr
947,soko fried rice,1.0,ktc pro mix vegetable,jagung manis kupas,0.285714,gr
